<h1>ASSIGNMENT 4 INSTRUCTIONS</h1>



Gradio is an open-source Python package that allows you to quickly build a demo or web application for your machine learning model, API, or any arbitrary Python function.

**This assignment accomplishes the following objectives:**
1. In Assignment 3, we used three key design principles (tool use, reflection, and planning) to create an agenticAI workflow for blogging: Planner(Groq) -> Researcher(OpenAI + web_search tool) -> Writer(OpenAI) -> Editor(Groq) -> Reviser(OpenAI).
2. In this assignment, we will learn how to use Gradio to convert our blogging workflow to a web application.
3. We will also make the output from each step in the workflow editable by the user (before the workflow is allowed to proceed to the next step) - thereby demonstrating a human-in-the-loop agenticAI workflow.

**In order to complete the assignment:**
1. Section 1: Assignment setup. Just read through to understand what's already available.
2. Section 2: Wrap the Blogging AgenticAI Workflow roles in functions. Complete coding steps 2.1 through 2.4.
3. Section 3: Create a Human-in-the-Loop Blogging AgenticAI Workflow using Gradio. Compelte coding steps 3.1 through 3.11.

**To submit:**
1. Rename the assignment as Assignment3-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas

# SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH)

### We'll first import the required libraries.

In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

### We'll now setup the API KEYS and Clients.

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

from google.colab import userdata
import os
import aisuite
from openai import OpenAI
from groq import Groq

# read the API_KEYs from Colab Secrets and expose it as an env variable
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# create the clients
openai_client = OpenAI()
groq_client = Groq()
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL


### The reponses we get back from LLMs aren't always well-formatted. We'll use this utility function to print LLM responses.

In [3]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# a nifty utility function to cleanly diplay your chat outputs
from IPython.display import display, HTML, Markdown

def show(title, text):
    display(HTML(f"""
     <h4>{title}</h4>
     <div style="white-space: pre-wrap; padding-left: 24px;">{text}</div>"""))

### Recall from Assignment 1 that an LLM only has direct access to the information it was trained on and is unaware of events that occurred after its knowledge cutoff date. As in Assignment 2, we can address this limitation by providing the OpenAI LLM access to OpenAI's built-in web_search tool.

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import json

# use OpenAI's built-in web_search tool to perform web_search
def openai_web_search(query: str):
  """
    Perform a web search using OpenAI's built-in web_search tool
    and return raw results.
  """
  resp = openai_client.responses.create(
      model=OPENAI_MODEL,
      input=query,
      tools=[{"type": "web_search"}],
  )
  return json.loads(resp.model_dump_json())

### The run_openai_query is identical to what we used in Assignment 2. It accepts the system prompt, the user prompt, and a list of tools that it can invoke as needed. And it returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API, invoke tools as needed;
# return response and query details (usage stats and tools invoked)
def run_openai_query(system_prompt, user_prompt, tools=[]):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Supports tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      tools=tools,
      max_turns=3,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details


### The run_groq_query is similar to the run_openai_query, with one important difference: it does not accept/invoke any tools. It returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API; DOES NOT support tools;
# return response and query details (usage stats and tools invoked)
def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Does not support tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      max_turns=1,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details

# SECTION 2: Wrap the Blogging AgenticAI Workflow Roles in Functions (COMPLETE STEPS 2.1 THROUGH 2.4)

### AgenticAI Workflow

In this section, we will wrap the Blogging AgenticAI Workflow roles in functions.

- def planner (topic: str) -> return plan
- def researcher (plan: str) -> return research
- def writer (plan: str, research: str) -> return draft
- def editor (plan: str, research: str, draft: str) -> return feedback
- def reviser (draft: str, feedback: str) -> return blog

### [No Coding: Planner Function]

The **planner** function has been defined for you (as an illustrative example):

def planner (topic: str) -> return plan

In [7]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.0: This code has already been converted to a function
#           def planner (topic: str) -> return plan
#           feel free to update the system and user prompts (not required)

# the planner takes a topic and returns a plan
def planner (topic: str):

  # planner system prompt
  planner_system_prompt = (
    f"""
    You are a planning agent for writing a blog post.
    Your job is to define structure and requirements, not to write prose.
    Be clear, concise, and structured.
    """
  )

  # planner user prompt
  planner_user_prompt = (
    f"""
    Blog topic:{topic}
    Create:
      1. A working blog title
      2. The target audience (1 sentence)
      3. The goal of the blog (1 sentence)
      4. A clear outline with 5-7 section headings
      5. A short research checklist of facts, statistics,
        or definitions that should be verified before writing
    Do not write the blog content.
    """
  )

  # use Groq (no tools) to create a plan
  plan, query_details = run_groq_query(planner_system_prompt, planner_user_prompt)

  # return the plan
  return plan

### [Complete Coding STEP 2.1: Researcher Function]:

Next, you'll define the **researcher** function:

def researcher (plan: str) -> return research

In [8]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.1: Convert this code to a function:
#           def researcher (plan: str) -> return research

def researcher(plan: str):

  # researcher system prompt
  researcher_system_prompt = (
    f"""
    You are a research assistant.
    Use web search to find accurate, up-to-date information.
    Do not write blog prose.
    Return factual notes with sources.
    """
  )

  # research user prompt
  researcher_user_prompt = (
    f"""
    Blog plan (outline + checklist):{plan}
    For each outline section:
      1. Provide 3-5 factual bullet points
      2. Include definitions or statistics where relevant
      3. Include a source for each section
    Return concise research notes only.
    """
  )

  # use OpenAI (with web_search) to do the research
  research, query_details = run_openai_query(researcher_system_prompt, researcher_user_prompt, tools=[openai_web_search])

  return research

### [Complete Coding STEP 2.2: Writer Function]:

Next, you'll define the **writer** function:

def writer (plan: str, research: str) -> return draft

In [9]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.2: Convert this code to a function:
#           def writer (plan: str, research: str) -> return draft

def writer(plan: str, research: str):

  # writer system prompt
  writer_system_prompt = (
    f"""
    You are a blog writer.
    Write clearly and engagingly for the target audience.
    Focus on structure, flow, and clarity.
    Do not critique your writing.
    """
  )

  # writer user prompt
  writer_user_prompt = (
    f"""
    Audience, goal, and outline from planner:{plan}
    Research and sources from researcher:{research}
    Write a complete one-page blog post.
    Use the audience, goal, and outline accurately.
    Use the research and sources accurately.
    """
  )

  # use OpenAI (no tools) to write a draft
  draft, query_details = run_openai_query(writer_system_prompt, writer_user_prompt, tools=[])

  return draft

### [Complete Coding STEP 2.3: Editor Function]:

Next, you'll define the **editor** function:

def editor (plan: str, research: str, draft: str) -> return feedback

In [10]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.3: Convert this code to a function:
#           def editor (plan: str, research: str, draft: str) -> return feedback

def editor(plan: str, research: str, draft: str):

  # editor system prompt
  editor_system_prompt = (
    f"""
    You are a critical editor.
    Your job is to evaluate and suggest improvements, not to rewrite the content.
    Be specific and constructive.
    """
  )

  # editor user prompt
  editor_user_prompt = (
    f"""
    Original audience, goal, and outline from planner:{plan}
    Research and sources from researcher:{research}
    Draft from writer:{draft}
    Evaluate on: clarity, structure/flow, accuracy (given the research), and tone.
    Provide:
      1. A 1-5 score for each category
      2. Specific, actionable improvement suggestions
    Do not rewrite the blog.
    """
  )

  # use Groq (no tools) provide feedback on the draft
  feedback, query_details = run_groq_query(editor_system_prompt, editor_user_prompt)

  return feedback

### [Complete Coding STEP 2.4: Reviser Function]:

Next, you'll define the **reviser** agent:

def reviser (draft: str, feedback: str) -> return blog

In [11]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.4: Convert this code to a function:
#           def reviser (draft: str, feedback: str) -> return blog

def reviser(draft: str, feedback: str):

  # reviser system prompt
  reviser_system_prompt = (
    f"""
    You are a blog writer revising a draft based on editorial feedback.
    Improve clarity, structure, and accuracy.
    Preserve the original intent and outline unless told otherwise.
    """
  )

  # reviser user prompt
  reviser_user_prompt = (
    f"""
    Original blog draft:{draft}
    Editor feedback:{feedback}
    Revise the blog post using the feedback.
    Return the full revised one-page blog post.
    """
  )

  # use OpenAI (no tools) creat the blog
  blog, query_details = run_openai_query(reviser_system_prompt, reviser_user_prompt, tools=[])

  return blog

# SECTION 3: Create a Human-in-the-Loop Blogging AgenticAI Workflow using Gradio

### [Complete Coding STEPS 3.1-3.11]:

Build a human-in-the-loop agenticAI workflow UI. Illustrative code for the first part of the UI (topic input, button that triggers planner, plan output, button that triggers researcher) has already been provided to you. I would suggest running this cell as-is first to see how this code shows up in the Gradio output. Use the public URL (that Gradio provides in the cell output) for a clearner view. Then, complete the code for the rest of the flow.
- Textbox to provide the topic input
- Button that triggers the planner when clicked
- Textbox to display the (editable) planner output
- When the submit topic button is clicked, call planner(topic) and fill plan (the plan is editable by user)
- Button that triggers the researcher when clicked
- CODING STEP-3.1: Textbox to display the (editable) researcher output
- CODING STEP-3.2: When the submit plan button is clicked, call researcher(plan) and fill research (the research is editable by user)
- CODING STEP-3.3: Button that triggers the writer when clicked
- CODING STEP-3.4: Textbox to display the (editable) writer output
- CODING STEP-3.5: When the submit research button is clicked, call writer(plan, research) and fill draft (the draft is editable by user)
- CODING STEP-3.6: Button that triggers the editor when clicked
- CODING STEP-3.7: Textbox to display the (editable) editor feedback
- CODING STEP-3.8: When the submit draft button is clicked, call editor(plan, research, draft) and fill feedback (the feedback is editable by user)
- CODING STEP-3.9: Button that triggers the reviser when clicked
- CODING STEP-3.10: Textbox to display the final output blog
- CODING STEP-3.11: When the submit feedback button is clicked, call reviser(draft, feedback) and fill blog

In [12]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

import gradio as gr

# build a human-in-the-loop agenticAI workflow UI
with gr.Blocks(title="Blog AgenticAI Workflow") as app:

    # Textbox to provide the topic input
    topic = gr.Textbox(label="Topic", placeholder="e.g. Artificial Intelligence")

    # Button that triggers the planner when clicked
    submit_topic_btn = gr.Button("Submit Topic → Run Planner")


    # Textbox to display the (editable) planner output
    plan = gr.Textbox(label="Plan from Planner (editable)", lines=10)

    # When the submit topic button is clicked, call planner(topic) and fill `plan`
    # The plan is editable by user
    submit_topic_btn.click(fn=planner, inputs=[topic], outputs=[plan])

    # Button that triggers the researcher when clicked
    submit_plan_btn = gr.Button("Submit Plan → Run Researcher")


    # STEP-3.1: Textbox to display the (editable) researcher output
    research = gr.Textbox(label="Research from Researcher (editable)", lines=10)

    # STEP-3.2: When the submit plan button is clicked, call researcher(plan) and fill `research`
    #           The research is editable by user
    submit_plan_btn.click(fn=researcher, inputs=[plan], outputs=[research])

    # STEP-3.3: Button that triggers the writer when clicked
    submit_research_btn = gr.Button("Submit Research → Run Writer")


    # STEP-3.4: Textbox to display the (editable) writer output
    draft = gr.Textbox(label="Draft from Writer (editable)", lines=10)

    # STEP-3.5: When the submit research button is clicked, call writer(plan, research) and fill `draft`
    #           The draft is editable by user
    submit_research_btn.click(fn=writer, inputs=[plan, research], outputs=[draft])

    # STEP-3.6: Button that triggers the editor when clicked
    submit_draft_btn = gr.Button("Submit Draft → Run Editor")


    # STEP-3.7: Textbox to display the (editable) editor feedback
    feedback = gr.Textbox(label="Feedback from Editor (editable)", lines=10)

    # STEP-3.8: When the submit draft button is clicked, call editor(plan, research, draft) and fill `feedback`
    #           The feedback is editable by user
    submit_draft_btn.click(fn=editor, inputs=[plan, research, draft], outputs=[feedback])

    # STEP-3.9: Button that triggers the reviser when clicked
    submit_feedback_btn = gr.Button("Submit Feedback → Run Reviser")


    # STEP-3.10: Textbox to display the final output blog
    blog = gr.Textbox(label="Final Blog from Reviser", lines=10)

    # STEP-3.11: When the submit feedback button is clicked, call reviser(draft, feedback) and fill `blog`
    submit_feedback_btn.click(fn=reviser, inputs=[draft, feedback], outputs=[blog])

# Start the Gradio web application
app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db49be9fb5b4889a63.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
